# Decision Tree from Scratch

A from-scratch implementation of a binary-split decision tree classifier
using **information gain** (entropy) as the splitting criterion. The tree
supports an optional `max_depth` cap and falls back gracefully when no
split yields positive gain.

The notebook compares my implementation against
`sklearn.tree.DecisionTreeClassifier` on Iris to confirm the splits,
structure, and held-out accuracy line up.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

np.random.seed(42)

## 2. Entropy and information gain

In [2]:
def entropy(y):
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))


def information_gain(y_parent, y_left, y_right):
    n = len(y_parent)
    if n == 0:
        return 0.0
    return (
        entropy(y_parent)
        - (len(y_left) / n) * entropy(y_left)
        - (len(y_right) / n) * entropy(y_right)
    )


y_toy = np.array(['Y', 'Y', 'N', 'Y', 'Y', 'N', 'N', 'N', 'N'])
print(f"entropy(toy 4 Y / 5 N) = {entropy(y_toy):.4f}")

entropy(toy 4 Y / 5 N) = 0.9911


## 3. `DecisionTree` classifier

In [3]:
class _Node:
    __slots__ = ("feature", "threshold", "left", "right", "prediction")

    def __init__(self, feature=None, threshold=None, left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction

    @property
    def is_leaf(self):
        return self.prediction is not None


class DecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.root = None
        self.feature_names_ = None
        self.classes_ = None

    def fit(self, X, y, feature_names=None):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.feature_names_ = (
            list(feature_names)
            if feature_names is not None
            else [f"x{i}" for i in range(X.shape[1])]
        )
        self.classes_ = np.unique(y)
        self.root = self._build(X, y, depth=0)
        return self

    def _majority(self, y):
        vals, counts = np.unique(y, return_counts=True)
        return vals[np.argmax(counts)]

    def _best_split(self, X, y):
        best_gain = 0.0
        best = None
        for j in range(X.shape[1]):
            values = np.unique(X[:, j])
            if len(values) < 2:
                continue
            thresholds = (values[:-1] + values[1:]) / 2.0
            for t in thresholds:
                left_mask = X[:, j] <= t
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                gain = information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best = (j, t, left_mask, right_mask)
        return best, best_gain

    def _build(self, X, y, depth):
        if len(np.unique(y)) == 1:
            return _Node(prediction=y[0])
        if self.max_depth is not None and depth >= self.max_depth:
            return _Node(prediction=self._majority(y))

        split, gain = self._best_split(X, y)
        if split is None or gain <= 0.0:
            return _Node(prediction=self._majority(y))

        j, t, left_mask, right_mask = split
        return _Node(
            feature=j,
            threshold=t,
            left=self._build(X[left_mask], y[left_mask], depth + 1),
            right=self._build(X[right_mask], y[right_mask], depth + 1),
        )

    def _predict_one(self, x, node):
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.prediction

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(x, self.root) for x in X])

    def print_tree(self, class_names=None):
        if self.root is None:
            print("(untrained tree)")
            return
        if self.root.is_leaf:
            label = class_names[self.root.prediction] if class_names is not None else self.root.prediction
            print(f"Class {label}")
            return
        lines = []
        self._gather(self.root, "", lines, class_names)
        print("\n".join(lines))

    def _gather(self, node, indent, lines, class_names):
        name = self.feature_names_[node.feature]
        t = node.threshold
        for child, op, next_indent in [
            (node.left,  "<=", indent + "|   "),
            (node.right, ">",  indent + "    "),
        ]:
            header = f"{indent}|-- [{name} {op} {t:g}]"
            if child.is_leaf:
                label = class_names[child.prediction] if class_names is not None else child.prediction
                lines.append(f"{header}: Class {label}")
            else:
                lines.append(header)
                self._gather(child, next_indent, lines, class_names)

## 4. ASCII visualization check

In [4]:
X_demo = np.array([
    [1.0, 1.0],
    [2.0, 1.0],
    [1.0, 2.0],
    [2.0, 2.0],
    [3.0, 3.0],
    [4.0, 5.0],
])
y_demo = np.array(['A', 'A', 'B', 'B', 'C', 'C'])

demo_tree = DecisionTree()
demo_tree.fit(X_demo, y_demo, feature_names=["Feature1", "Feature2"])
demo_tree.print_tree()

|-- [Feature1 <= 2.5]
|   |-- [Feature2 <= 1.5]: Class A
|   |-- [Feature2 > 1.5]: Class B
|-- [Feature1 > 2.5]: Class C


## 5. Iris, `max_depth=3`

In [5]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

my_tree = DecisionTree(max_depth=3)
my_tree.fit(X_train, y_train, feature_names=iris.feature_names)

my_tree.print_tree(class_names=iris.target_names)

y_pred = my_tree.predict(X_test)
my_acc = accuracy_score(y_test, y_pred)
print(f"\nTest accuracy: {my_acc:.4f}")

|-- [petal length (cm) <= 2.45]: Class setosa
|-- [petal length (cm) > 2.45]
    |-- [petal width (cm) <= 1.65]
    |   |-- [petal length (cm) <= 4.95]: Class versicolor
    |   |-- [petal length (cm) > 4.95]: Class virginica
    |-- [petal width (cm) > 1.65]
        |-- [petal length (cm) <= 4.85]: Class virginica
        |-- [petal length (cm) > 4.85]: Class virginica

Test accuracy: 0.9667


## 6. Comparison with `sklearn.tree.DecisionTreeClassifier`

In [6]:
sk_tree = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)
sk_tree.fit(X_train, y_train)

y_pred_sk = sk_tree.predict(X_test)
sk_acc = accuracy_score(y_test, y_pred_sk)

print(f"Test accuracy (from scratch): {my_acc:.4f}")
print(f"Test accuracy (sklearn):      {sk_acc:.4f}")
print(f"Predictions agree on {np.mean(y_pred == y_pred_sk) * 100:.1f}% of test points")
print()
print(export_text(sk_tree, feature_names=list(iris.feature_names)))

Test accuracy (from scratch): 0.9667
Test accuracy (sklearn):      0.9667
Predictions agree on 100.0% of test points

|--- petal length (cm) <= 2.45
|   |--- class: 0
|--- petal length (cm) >  2.45
|   |--- petal width (cm) <= 1.65
|   |   |--- petal length (cm) <= 4.95
|   |   |   |--- class: 1
|   |   |--- petal length (cm) >  4.95
|   |   |   |--- class: 2
|   |--- petal width (cm) >  1.65
|   |   |--- petal length (cm) <= 4.85
|   |   |   |--- class: 2
|   |   |--- petal length (cm) >  4.85
|   |   |   |--- class: 2



### Observations

Both trees pick the same feature and the same threshold at every split:
petal length $\le 2.45$ at the root, then petal width $\le 1.65$, then
petal length $\le 4.95$ on one side and petal length $\le 4.85$ on the
other. The thresholds match because on Iris the midpoints of consecutive
unique values (what I use) coincide with the midpoints sklearn uses
between adjacent sorted samples.

Structure also matches: depth 3, 3 internal nodes, same class labels at
the leaves. The last split, petal length $\le 4.85$, is degenerate (both
children are virginica). The algorithm still makes it because
`max_depth=3` forces expansion; the no-gain guard only stops splitting
earlier in the tree.

Both trees get **0.9667** on the held-out test set and agree on **100%**
of test points.